# Experiment: SpectralBio Final Accept Hardening Part 2

This notebook runs the heavier model-size and deep-checkpoint validation half of the final protocol.

## Scope
- TP53 stronger-backbone comparison
- Reuse or rebuild support-ranked panel artifacts from Part 1
- Deep checkpoint sweep on TP53 plus the strongest follow-up genes

## Intended runtime
- Best on L4
- Can run on Colab if `USE_3B = False`


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptSplit')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
REPO_ROOT = Path('/content/Stanford-Claw4s')
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v3_complete'

if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Restarting runtime once...')
    os.kill(os.getpid(), 9)
else:
    print('Bootstrap marker found; skipping reinstall.')


## Plan

- Start from the same support-ranked rules as Part 1.
- Run the heavy checkpoints only on the deepest validation genes.
- Set `USE_3B = False` on T4 if memory becomes tight.


In [ ]:
import subprocess
import sys
import pandas as pd

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        build_support_ranked_panel_manifest,
        create_accept_hardening_paths,
        recommend_second_benchmark_candidate,
        run_deep_checkpoint_sweep,
        run_support_ranked_panel,
        scan_clinvar_gene_support,
    )
    from spectralbio.supplementary.reject_recovery import (
        BackboneEvaluationConfig,
        create_reject_recovery_zip,
        run_tp53_backbone_comparison,
        write_experiment_log,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        build_support_ranked_panel_manifest,
        create_accept_hardening_paths,
        recommend_second_benchmark_candidate,
        run_deep_checkpoint_sweep,
        run_support_ranked_panel,
        scan_clinvar_gene_support,
    )
    from spectralbio.supplementary.reject_recovery import (
        BackboneEvaluationConfig,
        create_reject_recovery_zip,
        run_tp53_backbone_comparison,
        write_experiment_log,
    )

USE_3B = True
MIN_TOTAL = 60
MIN_PER_CLASS = 20
MAX_ADDITIONAL_GENES = 8
LARGE_MODEL_GENE_LIMIT = 4
BOOTSTRAP_REPLICATES = 1000
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_split'
)

model_names = ('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D') if USE_3B else ('facebook/esm2_t33_650M_UR50D',)
config = AcceptHardeningConfig(
    stronger_model_names=model_names,
    min_total=MIN_TOTAL,
    min_per_class=MIN_PER_CLASS,
    max_additional_genes=MAX_ADDITIONAL_GENES,
    large_model_gene_limit=LARGE_MODEL_GENE_LIMIT,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)
print(config)


In [ ]:
support_scan = scan_clinvar_gene_support(paths, config)
panel_manifest = build_support_ranked_panel_manifest(paths, config, support_scan)
multigene_metrics = run_support_ranked_panel(paths, config, panel_manifest)
candidate = recommend_second_benchmark_candidate(paths, config, panel_manifest, multigene_metrics)

tp53_backbone = run_tp53_backbone_comparison(
    paths,
    BackboneEvaluationConfig(
        reference_model_name=config.reference_model_name,
        stronger_model_name='facebook/esm2_t33_650M_UR50D',
        window_radius=config.window_radius,
        pair_alpha=config.pair_alpha,
        random_seed=config.random_seed,
        checkpoint_every=config.checkpoint_every,
        render_figures=config.render_figures,
        overwrite=config.overwrite,
        reuse_frozen_tp53_reference=config.reuse_frozen_tp53_reference,
        run_full_surface_alpha_sweep=True,
    ),
)

checkpoint_sweep = run_deep_checkpoint_sweep(paths, config, panel_manifest, multigene_metrics, candidate)
print('Recommended second benchmark candidate:', candidate['recommended_gene'])
print('Checkpoint sweep genes:', checkpoint_sweep['genes'])


In [ ]:
backbone_table = pd.read_csv(paths.backbone_strength / 'tp53_backbone_comparison.csv')
checkpoint_long = pd.read_csv(paths.root / 'checkpoint_sweep' / 'checkpoint_sweep_long.csv')
pair_pivot = pd.read_csv(paths.root / 'checkpoint_sweep' / 'checkpoint_sweep_pair_auc_pivot.csv')
delta_pivot = pd.read_csv(paths.root / 'checkpoint_sweep' / 'checkpoint_sweep_delta_pivot.csv')

display(backbone_table)
display(checkpoint_long.sort_values(['gene', 'model_rank']))
display(pair_pivot)
display(delta_pivot)


In [ ]:
notes = [
    'Part 2 focuses on backbone stress tests and deep checkpoint robustness.',
    f"Recommended second benchmark candidate: {candidate['recommended_gene']}",
    f"USE_3B was set to {USE_3B}",
]
log_path = write_experiment_log(
    paths,
    completed_experiments=[
        'Support-ranked panel load or rebuild',
        'TP53 650M backbone comparison',
        'Deep checkpoint sweep',
    ],
    skipped_experiments=[],
    notes=notes,
)
zip_path = create_reject_recovery_zip(paths, bundle_name='spectralbio_final_accept_part2_bundle')
print('Experiment log:', log_path)
print('ZIP ready at:', zip_path)


In [ ]:
print(f'ZIP ready at: {zip_path}')

if zip_path.exists():
    from google.colab import files
    files.download(str(zip_path))
else:
    print('ZIP not found:', zip_path)
